# Tutorial 4 - Live Excel Batch Simulation

This tutorial shows the full Excel simulation workflow. Offline cells use a fake client; live cells submit to BRAIN only when `BRAIN_SIM_RUN_LIVE=1`.


In [ ]:
from __future__ import annotations

import csv
import json
import os
import shutil
import sqlite3
import subprocess
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Any

RUN_LIVE = os.getenv("BRAIN_SIM_RUN_LIVE") == "1"
CWD = Path.cwd().resolve()
ROOT = CWD if (CWD / "examples").exists() else CWD.parent if CWD.name == "examples" else CWD
EXAMPLE_DIR = ROOT / "examples"
DATA_DIR = EXAMPLE_DIR / "data"
EXPECTED_DIR = EXAMPLE_DIR / "expected"
RUNS_DIR = EXAMPLE_DIR / "runs"
RUNS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Repository root: {ROOT}")
print(f"Live execution enabled: {RUN_LIVE}")


In [ ]:
@dataclass(frozen=True)
class TutorialRateLimitState:
    limit: int | None
    remaining: int | None
    reset_seconds: int | None


@dataclass(frozen=True)
class TutorialSubmitResult:
    status_code: int
    location: str
    body: Any
    headers: dict[str, str]
    rate_limit: TutorialRateLimitState


@dataclass(frozen=True)
class TutorialPollResult:
    status: str
    body: Any
    events: list[dict[str, Any]]


class CompleteFakeBrainClient:
    def __init__(self, alpha_prefix: str = "tutorial-alpha", location_prefix: str = "/simulations/tutorial") -> None:
        self.alpha_prefix = alpha_prefix
        self.location_prefix = location_prefix
        self.submit_count = 0
        self.locations: dict[str, list[str]] = {}

    def submit(self, payload: dict[str, Any] | list[dict[str, Any]]) -> TutorialSubmitResult:
        self.submit_count += 1
        payloads = payload if isinstance(payload, list) else [payload]
        alpha_ids = [f"{self.alpha_prefix}-{self.submit_count + offset}" for offset in range(len(payloads))]
        location = f"{self.location_prefix}-{self.submit_count}"
        self.locations[location] = alpha_ids
        return TutorialSubmitResult(
            status_code=201,
            location=location,
            body={"id": location.rsplit("/", 1)[-1]},
            headers={"Location": location, "X-Ratelimit-Remaining": "999"},
            rate_limit=TutorialRateLimitState(limit=None, remaining=999, reset_seconds=None),
        )

    def poll(self, location: str, *, timeout_seconds: float) -> TutorialPollResult:
        alpha_ids = self.locations[location]
        body: Any = {"alpha": alpha_ids[0]} if len(alpha_ids) == 1 else {"alphas": alpha_ids}
        return TutorialPollResult(status="complete", body=body, events=[{"body": body}])

    def fetch_alpha(self, alpha_id: str) -> dict[str, Any]:
        index = int(alpha_id.rsplit("-", 1)[-1])
        return {
            "id": alpha_id,
            "status": "UNSUBMITTED",
            "is": {
                "sharpe": f"{1.0 + index / 10:.2f}",
                "fitness": f"{0.5 + index / 10:.2f}",
                "returns": f"{0.03 + index / 100:.2f}",
                "turnover": f"{0.10 + index / 100:.2f}",
                "drawdown": f"{0.02 + index / 100:.2f}",
                "checks": [
                    {"name": "LOW_SHARPE", "result": "WARNING"} if index == 2 else {"name": "SELF_CORRELATION", "result": "PASS"}
                ],
            },
        }

    def fetch_recordset(self, alpha_id: str, recordset_name: str) -> dict[str, Any]:
        return {
            "alpha_id": alpha_id,
            "recordset": recordset_name,
            "rows": [
                {"date": "2026-01-01", "value": 0.01},
                {"date": "2026-01-02", "value": 0.02},
            ],
        }


## 1. Build Payloads From Excel

The same workbook can be used for offline rehearsal and live `simulate-excel` runs.


In [ ]:
from brain_sim.excel import read_excel_expressions
from brain_sim.payloads import build_payload_record

excel_path = DATA_DIR / "tutorial_04_live_alphas.xlsx"
alpha_rows = read_excel_expressions(excel_path, sheet_name="alphas")
payload_records = []
for index, alpha in enumerate(alpha_rows, start=1):
    record = build_payload_record(alpha)
    payload_records.append(
        record.__class__(
            row_id=record.row_id,
            alpha_hash=f"tutorial-hash-{index}",
            payload=record.payload,
            metadata=record.metadata,
        )
    )

print(f"Loaded {len(payload_records)} payload records from {excel_path.name}")
[(record.row_id, record.alpha_hash, record.payload["settings"]["universe"]) for record in payload_records]


## 2. Offline Batch Run

The offline run writes the same artifact shape as a live run: raw logs, alpha detail JSON, summary CSV, retry queue, and SQLite cache.


In [ ]:
from brain_sim.batch import BatchRunner

run_dir = RUNS_DIR / "tutorial_04_offline_batch"
shutil.rmtree(run_dir, ignore_errors=True)
runner = BatchRunner(CompleteFakeBrainClient(), run_dir)
result = runner.run(payload_records, batch_size=4, poll_timeout_seconds=5)
print(result)

summary_path = run_dir / "summary.csv"
with summary_path.open(newline="", encoding="utf-8") as f:
    summary_rows = list(csv.DictReader(f))
summary_rows


## 3. Live CLI Commands

These commands consume live BRAIN simulation quota. Run them only after `brain-sim login` has saved cookies.


In [ ]:
live_commands = [
    ["brain-sim", "simulate-excel", str(DATA_DIR / "tutorial_04_live_alphas.xlsx"), "--sheet", "alphas", "--batch-size", "auto", "--poll-timeout-seconds", "1800"],
    ["brain-sim", "simulate-excel", str(DATA_DIR / "tutorial_04_live_alphas.xlsx"), "--sheet", "alphas", "--batch-size", "8"],
    ["brain-sim", "simulate-excel", str(DATA_DIR / "tutorial_04_live_alphas.xlsx"), "--sheet", "alphas", "--batch-size", "4"],
    ["brain-sim", "simulate-excel", str(DATA_DIR / "tutorial_04_live_alphas.xlsx"), "--sheet", "alphas", "--batch-size", "1"],
]

if RUN_LIVE:
    subprocess.run(live_commands[0], cwd=ROOT, check=False)
else:
    for command in live_commands:
        print(" ".join(command))
